In [5]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 假设 test_df 已经是你的测试集，包含 'time' 和 'UVI'
uvi_series = test_df["UVI"].values

seq_len = 96   # 历史输入长度
pred_len = 24  # 预测长度

y_true = []
y_pred = []

# 构造每个测试样本的Persistence预测
for i in range(len(uvi_series) - seq_len - pred_len + 1):
    # 真实未来24小时
    future = uvi_series[i+seq_len:i+seq_len+pred_len]
    
    # 最近24小时作为预测
    persistence = uvi_series[i+seq_len-pred_len:i+seq_len]
    
    y_true.append(future)
    y_pred.append(persistence)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Shape check:", y_true.shape, y_pred.shape)  # 应该都是 (N, 24)

# 计算指标
mae = mean_absolute_error(y_true.flatten(), y_pred.flatten())
rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
r2 = r2_score(y_true.flatten(), y_pred.flatten())

print("Persistence Baseline")
print(f"MAE  = {mae:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"R²   = {r2:.4f}")

Shape check: (2022, 24) (2022, 24)
Persistence Baseline
MAE  = 0.3215
RMSE = 0.8649
R²   = 0.7554


In [4]:
df = pd.read_csv("final_dataset_hourly.csv", index_col=0, parse_dates=True)

# 填充 UVI > 20 为 NaN
df.loc[df["UVI"] > 20, "UVI"] = np.nan
# 线性插值补全
df = df.interpolate(method="linear")
# 夜间缺失的可填 0
df["UVI"].fillna(0, inplace=True)

# 周期特征编码
df['hour_sin'] = np.sin(2*np.pi*df.index.hour/24)
df['hour_cos'] = np.cos(2*np.pi*df.index.hour/24)
df['day_year_sin'] = np.sin(2*np.pi*df.index.dayofyear/365)
df['day_year_cos'] = np.cos(2*np.pi*df.index.dayofyear/365)

feature_cols = ['temp','rainfall','windspd','windspd_max','wind_d',
                'GHI','DNI','DHI','UVA','ClearSkyGHI','CSI',
                'hour_sin','hour_cos','day_year_sin','day_year_cos']
target_col = 'UVI'

# 数据集切分
train_df = df.loc['2020-01-01':'2021-06-30']
val_df   = df.loc['2021-07-01':'2021-09-30']
test_df  = df.loc['2021-10-01':'2021-12-29']

C:\Users\apple\AppData\Local\Temp\ipykernel_11144\4134516513.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["UVI"].fillna(0, inplace=True)


In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import math

C:\Users\apple\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [6]:
time_index = []

for i in range(len(y_pred)):
    start = test_df.index[i + 96]
    rng = pd.date_range(
        start=start,
        periods=24,
        freq='H'
    )
    time_index.extend(rng)

len(time_index)

C:\Users\apple\AppData\Local\Temp\ipykernel_11144\1364800189.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  rng = pd.date_range(


48528

In [7]:
export_df = pd.DataFrame({
    "time": time_index,
    "True_UVI": y_true.flatten(),
    "Persistence_Pred": y_pred.flatten()
})

export_df.to_csv(
    "Persistence_with_time.csv",
    index=False
)